<a href="https://colab.research.google.com/github/madhesh60/BPE/blob/main/TokenizerBPE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset
from collections import defaultdict, Counter
import regex as re  # For GPT-2 style pretokenization

In [ ]:
# Load UltraChat 200k - use train_sft split for tokenizer training [[29]]
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True)

# Extract text from messages (user + assistant content)
def extract_text(example):
    texts = []
    for msg in example["messages"]:
        if msg["content"]:
            texts.append(msg["content"])
    return " ".join(texts)

    def text_generator(dataset, max_samples=100_000):
       for i, example in enumerate(dataset):
        if i >= max_samples:
            break
        yield extract_text(example)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

In [ ]:
class BPETokenizer:
    def __init__(self, vocab_size=32000):
        self.vocab_size = vocab_size
        self.merges = {}
        self.vocab = {}
        self.id_to_token = {}

    def _bytes_to_unicode(self):
        bs = list(range(ord("!"), ord("~")+1)) + list(range(ord("¡"), ord("¬")+1)) + list(range(ord("®"), ord("ÿ")+1))
        cs = bs[:]
        n = 0
        for b in range(2**8):
            if b not in bs:
                bs.append(b)
                cs.append(2**8 + n)
                n += 1
        return {b: chr(c) for b, c in zip(bs, cs)}

    def _get_stats(self, tokens):
        pairs = Counter()
        for token_seq in tokens:
            for i in range(len(token_seq) - 1):
                pair = (token_seq[i], token_seq[i + 1])
                pairs[pair] += 1
        return pairs

    def _merge_pair(self, a, b, new_id, tokens):
        new_tokens = []
        for token_seq in tokens:
            new_seq = []
            i = 0
            while i < len(token_seq):
                if i < len(token_seq) - 1 and token_seq[i] == a and token_seq[i+1] == b:
                    new_seq.append(new_id)
                    i += 2
                else:
                    new_seq.append(token_seq[i])
                    i += 1
            new_tokens.append(new_seq)
        return new_tokens

    def train(self, text_iterator, pretokenize_pattern=None):
        self.byte_to_char = self._bytes_to_unicode()
        self.char_to_byte = {v: k for k, v in self.byte_to_char.items()}

        for i in range(256):
            char = self.byte_to_char[i]
            self.vocab[char] = i
            self.id_to_token[i] = char

        if pretokenize_pattern is None:
            pretokenize_pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
        pretokenizer = re.compile(pretokenize_pattern)

        all_tokens = []
        for text in text_iterator:
            chunks = pretokenizer.findall(text)
            for chunk in chunks:
                token_ids = list(chunk.encode('utf-8'))
                all_tokens.append(token_ids)

        next_id = 256
        while next_id < self.vocab_size:
            pair_freqs = self._get_stats(all_tokens)
            if not pair_freqs: break
            (a, b), _ = pair_freqs.most_common(1)[0]

            new_token_str = self.id_to_token[a] + self.id_to_token[b]
            self.merges[(a, b)] = next_id
            self.vocab[new_token_str] = next_id
            self.id_to_token[next_id] = new_token_str

            all_tokens = self._merge_pair(a, b, next_id, all_tokens)
            next_id += 1
            if next_id % 1000 == 0:
                print(f"Trained {next_id}/{self.vocab_size} tokens")

    def encode(self, text, pretokenize_pattern=None):
        if pretokenize_pattern is None:
            pretokenize_pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
        pretokenizer = re.compile(pretokenize_pattern)
        tokens = []
        chunks = pretokenizer.findall(text)
        for chunk in chunks:
            token_ids = list(chunk.encode('utf-8'))
            for (a, b), new_id in self.merges.items():
                i = 0
                while i < len(token_ids) - 1:
                    if token_ids[i] == a and token_ids[i+1] == b:
                        token_ids = token_ids[:i] + [new_id] + token_ids[i+2:]
                    else:
                        i += 1
            tokens.extend(token_ids)
        return tokens

    def decode(self, token_ids):
        full_string = "".join([self.id_to_token[idx] for idx in token_ids])
        byte_array = bytearray([self.char_to_byte[char] for char in full_string])
        return byte_array.decode('utf-8', errors='replace')

In [ ]:
# Initialize and train
tokenizer = BPETokenizer(vocab_size=32000)

def text_generator(dataset, max_samples=100_000):
       for i, example in enumerate(dataset):
        if i >= max_samples:
            break
        yield extract_text(example)

# Train on first 50k examples (adjust based on compute)
tokenizer.train(
    text_generator(dataset, max_samples=10_000),
    pretokenize_pattern=r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

# Test encoding/decoding
test_text = "Hello! How are you doing today? 🤗"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)
print(f"Original: {test_text}")
print(f"Encoded: {encoded[:20]}... ({len(encoded)} tokens)")
print(f"Decoded: {decoded}")
print(f"Round-trip OK: {test_text == decoded}")